# InsureRAG-VLM End-to-End Demo

This notebook demonstrates how to build the retrieval index, query the VLM pipeline, run evaluation, compare clause differences, and extract PDF pages.

## 1. Import Required Libraries

Import the pipeline and helper modules for retrieval, OCR, evaluation, diff analysis, and PDF extraction.

In [ ]:
from pathlib import Path
import os

from src.insurerag_vlm.config import ModelConfig
from src.insurerag_vlm.pipeline import DocumentRetrievalPipeline
from src.insurerag_vlm.diff import compare_clause_diff, render_clause_diff, summarize_clause_diff
from src.insurerag_vlm.evaluation import generate_evaluation_examples
from src.insurerag_vlm.pdf import extract_text_by_page, extract_layout_by_page


## 2. Build the Retrieval Index

Build a text/image/PDF retrieval index from a data folder. Set the proper API keys before running this cell.

In [ ]:
data_folder = Path('data')
config = ModelConfig(
    use_hf_api=False,
    openai_api_key=os.getenv('OPENAI_API_KEY'),
    hf_api_token=os.getenv('HF_API_TOKEN'),
)
pipeline = DocumentRetrievalPipeline(config)
pipeline.build_index(data_folder)
print('Index built successfully:', config.index_path.exists(), config.metadata_path.exists())

## 3. Query the Pipeline with Page Ranking

Retrieve the top candidate pages and generate a grounded answer from the VLM.

In [ ]:
question = 'What coverage does the endorsement provide?'
result = pipeline.query_with_ranking(question, data_folder, top_k=5)
print('Answer:\n', result['answer'])
print('\nTop ranked pages:')
for candidate in result['source_ranking']:
    print(f"- {candidate['source']} (score={candidate['score']:.4f})")


## 4. Evaluate QA Predictions

Run evaluation over a JSON file of question-answer examples to compute EM, F1, and citation precision.

In [ ]:
examples_path = Path('examples.json')
if examples_path.exists():
    metrics = pipeline.evaluate(data_folder, examples_path, top_k=5)
    print('Evaluation metrics:')
    for name, value in metrics.items():
        print(f'{name}: {value:.4f}')
else:
    print('examples.json not found; please create a dataset to run evaluation.')

In [ ]:
<VSCode.Cell id="#VSC-db05679d" language="markdown">
## 4.5 Generate Evaluation Dataset

Convert QA input files into the JSON evaluation format used by the pipeline.
</VSCode.Cell>

## 5. Clause Diff with Sentence-Level Scoring

Compare two documents and inspect the diff summary and top sentence changes.

In [ ]:
original_path = Path('policy_v1.txt')
revised_path = Path('policy_v2.txt')
if original_path.exists() and revised_path.exists():
    old_text = original_path.read_text(encoding='utf-8', errors='ignore')
    new_text = revised_path.read_text(encoding='utf-8', errors='ignore')
    changes = compare_clause_diff(old_text, new_text)
    print('Clause diff:')
    print(render_clause_diff(changes)[:2000])
    print('\nDiff summary:')
    print(summarize_clause_diff(old_text, new_text))
else:
    print('policy_v1.txt or policy_v2.txt not found; add sample files to compare.')


## 6. PDF Extraction and Layout

Show how to extract PDF text and layout blocks using the built-in helpers.

In [ ]:
pdf_path = Path('policy.pdf')
if pdf_path.exists():
    page_texts = extract_text_by_page(pdf_path)
    print(f'Extracted {len(page_texts)} pages from PDF.')
    print('Page 1 preview:')
    print(page_texts[0][:800])
    layouts = extract_layout_by_page(pdf_path)
    print(f'Page layout blocks on page 1: {len(layouts[0].blocks)}')
else:
    print('policy.pdf not found; add a PDF file to run extraction.')